In [ ]:
import pandas as pd
from pytrends.request import TrendReq
import time
import numpy as np
import geopandas as gpd

In [ ]:
pytrends = TrendReq(hl='en-US', tz=360)

In [ ]:
# List of all 30 NBA Teams
nba_teams = [
    "Atlanta Hawks", "Boston Celtics", "Brooklyn Nets", "Charlotte Hornets", "Chicago Bulls",
    "Cleveland Cavaliers", "Dallas Mavericks", "Denver Nuggets", "Detroit Pistons", "Golden State Warriors",
    "Houston Rockets", "Indiana Pacers", "LA Clippers", "Los Angeles Lakers", "Memphis Grizzlies",
    "Miami Heat", "Milwaukee Bucks", "Minnesota Timberwolves", "New Orleans Pelicans", "New York Knicks",
    "Oklahoma City Thunder", "Orlando Magic", "Philadelphia 76ers", "Phoenix Suns", "Portland Trail Blazers",
    "Sacramento Kings", "San Antonio Spurs", "Toronto Raptors", "Utah Jazz", "Washington Wizards"
]

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
pd.set_option("display.max_rows", 210)

In [ ]:
def get_nba_fandom_anchor(timeframe='today 5-y', anchor_team="Golden State Warriors"):
    all_dfs = []
    benchmark = anchor_team
    
    # Process in chunks of 4 teams + 1 benchmark = 5 max keywords
    for i in range(0, len(nba_teams), 4):
        chunk = nba_teams[i:i + 4]
        kw_list = [benchmark] + chunk
        
        success = False
        retries = 0
        wait_time = 60  # Base wait time in seconds if an error occurs
        
        # Retry loop for error handling
        while not success:
            try:
                print(f"Fetching data for: {chunk}...")
                
                # Build payload with an expanded 5-year timeframe
                pytrends.build_payload(kw_list, timeframe=timeframe, geo='US')
                dma_data = pytrends.interest_by_region(resolution='DMA', inc_low_vol=True)
                
                # Express as percentage of anchor
                dma_data = dma_data.astype(float)
                dma_data.iloc[:, 1:] = dma_data.iloc[:, 1:].div(dma_data.iloc[:, 0], axis=0)

                # Keep the benchmark column ONLY on the first chunk to anchor the data
                # Drop benchmark column
                if i > 0:
                    dma_data = dma_data.drop(columns=[benchmark])
                else:
                    dma_data.iloc[:, 0] = 1.0

                
                all_dfs.append(dma_data)
                success = True  # Break the while loop and move to next chunk
                
                # Standard polite delay between successful requests
                print("Success. Sleeping for 10 seconds to respect rate limits...\n")
                time.sleep(10)
                
            except Exception as e:
                retries += 1
                print(f"\n[!] Error encountered: {e}")
                print(f"[!] Rate limit or connection issue. Sleeping for {wait_time} seconds (Attempt {retries})...")
                
                time.sleep(wait_time)
                wait_time *= 2  # Exponential backoff (60s, 120s, 240s...)
                
                # Optional safety valve to prevent infinite loops
                if retries > 5:
                    print("Too many consecutive failures. Saving what we have and exiting.")
                    # Return what we've collected so far before crashing
                    return pd.concat(all_dfs, axis=1) if all_dfs else None

    # Concatenate all chunks into a single matrix
    master_df = pd.concat(all_dfs, axis=1)
    return master_df


In [ ]:
bc_anchor_df = get_nba_fandom_anchor(timeframe="today 5-y", anchor_team="Boston Celtics")

In [ ]:
# now normalize all these to percentages
def normalize_trends(trends_df: pd.DataFrame):
    # create copy
    df = trends_df.copy().astype(float)

    # convert to floats
    df = df.astype(float)

    # sum over the rows
    row_sums = df.sum(axis=1)

    # get the percentages
    df = df.divide(row_sums, axis=0).fillna(0)

    return df

In [ ]:
final_fandom_df = normalize_trends(bc_anchor_df)

In [ ]:
final_fandom_df.head()

In [ ]:
final_fandom_df.loc["New York NY"]

In [ ]:
final_fandom_df.to_csv("../data/source_data/nba_team_fandom_5yr.csv")